In [3]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import json
import os
import time
from scipy.stats import wilcoxon
import scikit_posthocs as sp

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              log_loss)
from xgboost import XGBClassifier

# ============================================================
# SECTION 0: CONFIGURATION
# ============================================================
RANDOM_SEED    = 42
TEST_SIZE      = 0.2
OUTPUT_DIR     = "Single Train-Test ABB_results"
N_BOOTSTRAP    = 1000          # bootstrap iterations for Wilcoxon APFD test
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(RANDOM_SEED)

# ============================================================
# SECTION 1: DATA LOADING
# ============================================================
print("=" * 60)
print("SECTION 1: Loading Data")
print("=" * 60)

df = pd.read_csv(r'C:\Users\user\Desktop\BASE 1\tcp.csv')
print(f"Raw shape: {df.shape}")

# ============================================================
# SECTION 2: PREPROCESSING
# ============================================================
print("=" * 60)
print("SECTION 2: Preprocessing")
print("=" * 60)

df.drop(columns=[c for c in ['Id', 'Name', 'CalcPrio'] if c in df.columns],
        inplace=True)
df.dropna(subset=['Verdict'], inplace=True)
df.fillna(method='ffill', inplace=True)
df.drop_duplicates(inplace=True)

print(f"After cleaning: {df.shape}")

if 'Duration' in df.columns:
    df['Duration_log'] = np.log1p(df['Duration'])
    df.drop(columns=['Duration'], inplace=True)
    print("Duration log-transformed.")

print(f"\nClass distribution:\n{df['Verdict'].value_counts()}")
print(f"Fail rate: {df['Verdict'].mean()*100:.1f}%\n")

# ============================================================
# SECTION 3: FEATURE SETUP
# ============================================================
print("=" * 60)
print("SECTION 3: Feature Setup")
print("=" * 60)

X = df.drop('Verdict', axis=1)
y = df['Verdict'].astype(int)

numeric_features     = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features:     {numeric_features}")
print(f"Categorical features: {categorical_features}")

# ============================================================
# SECTION 4: TRAIN-TEST SPLIT  (80 / 20 stratified)
# ============================================================
print("=" * 60)
print("SECTION 4: Train-Test Split (80/20 Stratified)")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE,
    random_state=RANDOM_SEED, stratify=y
)

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")
print(f"Train fail rate : {y_train.mean()*100:.1f}%")
print(f"Test  fail rate : {y_test.mean()*100:.1f}%\n")

# ============================================================
# SECTION 5: PREPROCESSOR
# ============================================================
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                          numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'),    categorical_features)
])

# ============================================================
# SECTION 6: APFD FUNCTION
# ============================================================
def compute_apfd(y_true, y_prob):
    df_tmp = pd.DataFrame({'y': y_true, 'prob': y_prob})
    df_tmp = df_tmp.sort_values('prob', ascending=False).reset_index(drop=True)
    n = len(df_tmp)
    m = int(df_tmp['y'].sum())
    if m == 0:
        return 0.0
    fp = df_tmp.index[df_tmp['y'] == 1].to_numpy() + 1
    return float(1 - fp.sum()/(n*m) + 1/(2*n))

# ============================================================
# SECTION 7: MODEL PIPELINES
# ============================================================
print("=" * 60)
print("SECTION 7: Defining Models")
print("=" * 60)

def make_pipelines():
    return {
        "MLP": Pipeline([
            ('prep',  preprocessor),
            ('model', MLPClassifier(max_iter=300, random_state=RANDOM_SEED))
        ]),
        "RF": Pipeline([
            ('prep',  preprocessor),
            ('model', RandomForestClassifier(random_state=RANDOM_SEED,
                                             class_weight='balanced'))
        ]),
        "XGB": Pipeline([
            ('prep',  preprocessor),
            ('model', XGBClassifier(eval_metric='logloss',
                                    random_state=RANDOM_SEED,
                                    use_label_encoder=False))
        ]),
        "LR": Pipeline([
            ('prep',  preprocessor),
            ('model', LogisticRegression(max_iter=1000,
                                         random_state=RANDOM_SEED,
                                         class_weight='balanced'))
        ]),
        "SVM": Pipeline([
            ('prep',  preprocessor),
            ('model', SVC(probability=True, random_state=RANDOM_SEED,
                          class_weight='balanced'))
        ]),
        "KNN": Pipeline([
            ('prep',  preprocessor),
            ('model', KNeighborsClassifier())
        ])
    }

pipelines = make_pipelines()

# ============================================================
# SECTION 8: HYPERPARAMETER GRIDS
# ============================================================
param_grids = {
    "MLP": {
        "model__hidden_layer_sizes": [(64,32),(128,64),(128,64,32)],
        "model__activation":         ['relu','tanh'],
        "model__learning_rate_init": [0.001, 0.01, 0.1]
    },
    "RF":  {"model__n_estimators":[100,200], "model__max_depth":[None,10,20]},
    "XGB": {"model__n_estimators":[100,200], "model__max_depth":[3,5],
            "model__learning_rate":[0.01,0.1]},
    "LR":  {"model__C":[0.01,0.1,1.0]},
    "SVM": {"model__C":[0.1,1.0], "model__kernel":['rbf']},
    "KNN": {"model__n_neighbors":[3,5,7]}
}

# ============================================================
# SECTION 9: TRAINING WITH GRIDSEARCHCV
# ============================================================
print("=" * 60)
print("SECTION 9: Training with GridSearchCV (cv=5 on train set)")
print("=" * 60)

best_models     = {}
best_params_all = {}
results         = {}
apfd_scores     = {}

for name in pipelines:
    print(f"\nTraining {name}...")

    grid = GridSearchCV(
        estimator  = pipelines[name],
        param_grid = param_grids[name],
        cv=5, scoring='f1', n_jobs=-1, refit=True, verbose=0
    )
    grid.fit(X_train, y_train)

    best_models[name]     = grid.best_estimator_
    best_params_all[name] = grid.best_params_
    print(f"  Best params: {grid.best_params_}")

    y_pred = grid.best_estimator_.predict(X_test)
    y_prob = grid.best_estimator_.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    apfd = compute_apfd(y_test.values, y_prob)

    results[name]    = dict(accuracy=acc, precision=prec,
                            recall=rec, f1=f1, apfd=apfd)
    apfd_scores[name] = apfd

    print(f"  Accuracy={acc:.4f} | Precision={prec:.4f} | "
          f"Recall={rec:.4f} | F1={f1:.4f} | APFD={apfd:.4f}")
# ============================================================
# SECTION 10: RESULTS SUMMARY – DEFAULT PARAMETERS (Pre-Tuning)
# ============================================================
print("\n" + "=" * 60)
print("SECTION 10: Results Summary – Default Parameters (Pre-Tuning)")
print("=" * 60)

default_pipelines = make_pipelines()
default_results   = {}

for name, pipe in default_pipelines.items():
    print(f"  Fitting {name} with default params …", end=" ")
    pipe.fit(X_train, y_train)
    y_pred_d = pipe.predict(X_test)
    y_prob_d = pipe.predict_proba(X_test)[:, 1]

    default_results[name] = dict(
        accuracy  = accuracy_score(y_test, y_pred_d),
        precision = precision_score(y_test, y_pred_d, zero_division=0),
        recall    = recall_score(y_test, y_pred_d, zero_division=0),
        f1        = f1_score(y_test, y_pred_d, zero_division=0),
        apfd      = compute_apfd(y_test.values, y_prob_d)
    )
    print("done")

# Build summary table
default_rows = []
for name in default_pipelines:
    r = default_results[name]
    default_rows.append([name,
        f"{r['accuracy']:.4f}", f"{r['precision']:.4f}",
        f"{r['recall']:.4f}",  f"{r['f1']:.4f}", f"{r['apfd']:.4f}"])

default_df = pd.DataFrame(default_rows,
    columns=["Model","Accuracy","Precision","Recall","F1-Score","APFD"])

# Bold the best value in each column
for col in ["Accuracy","Precision","Recall","F1-Score","APFD"]:
    vals = default_df[col].astype(float)
    idx  = vals.idxmax()
    default_df.loc[idx, col] = "**" + default_df.loc[idx, col] + "**"

print("\n  Default-Parameter Results:")
print(default_df.to_string(index=False))
default_df.to_csv(f"{OUTPUT_DIR}/table_default_results.csv", index=False)
print("  Saved: table_default_results.csv")

# ============================================================
# SECTION 11: RESULTS SUMMARY TABLE
# ============================================================
print("\n" + "=" * 60)
print("SECTION 10: Results Summary")
print("=" * 60)

summary_rows = []
for name in pipelines:
    r = results[name]
    summary_rows.append([name,
        f"{r['accuracy']:.4f}", f"{r['precision']:.4f}",
        f"{r['recall']:.4f}",  f"{r['f1']:.4f}", f"{r['apfd']:.4f}"])

table3_df = pd.DataFrame(summary_rows,
    columns=["Model","Accuracy","Precision","Recall","F1-Score","APFD"])

for col in ["Accuracy","Precision","Recall","F1-Score","APFD"]:
    vals = table3_df[col].astype(float)
    idx  = vals.idxmax()
    table3_df.loc[idx, col] = "**" + table3_df.loc[idx, col] + "**"

print(table3_df.to_string(index=False))
table3_df.to_csv(f"{OUTPUT_DIR}/table3_results.csv", index=False)

# ============================================================
# SECTION 12: CONFUSION MATRICES
# ============================================================
print("\n" + "=" * 60)
print("SECTION 11: Confusion Matrices")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, name in enumerate(pipelines):
    y_pred = best_models[name].predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(cm, display_labels=['Pass','Fail'])
    disp.plot(ax=axes[i], colorbar=False)
    axes[i].set_title(f"{name} Confusion Matrix", fontsize=13)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrices.png", dpi=300)
plt.close()
print("Saved: confusion_matrices.png")

# ============================================================
# SECTION 13: APFD BAR CHART
# ============================================================
plt.figure(figsize=(10,6))
model_names = list(pipelines.keys())
apfd_vals   = [apfd_scores[n] for n in model_names]

bars = plt.bar(model_names, apfd_vals, color='steelblue', edgecolor='black')
plt.ylabel("APFD Score");  plt.title("APFD Comparison Across Models")
plt.ylim(0, 1.1)
for bar, val in zip(bars, apfd_vals):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f"{val:.4f}", ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/apfd_bar_chart.png", dpi=300)
plt.close()
print("Saved: apfd_bar_chart.png")

# ============================================================
# SECTION 14: METRICS BAR CHART
# ============================================================
metric_names = ["Accuracy","Precision","Recall","F1-Score","APFD"]
metric_keys  = ["accuracy","precision","recall","f1","apfd"]
x     = np.arange(len(metric_names));  width = 0.13

fig, ax = plt.subplots(figsize=(14,7))
for i, name in enumerate(pipelines):
    vals = [results[name][k] for k in metric_keys]
    ax.bar(x + i*width, vals, width, label=name)

ax.set_xlabel("Metric");  ax.set_ylabel("Score")
ax.set_title("Performance Metrics per Model")
ax.set_xticks(x + width*2.5);  ax.set_xticklabels(metric_names)
ax.legend();  ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/metrics_bar_chart.png", dpi=300)
plt.close()
print("Saved: metrics_bar_chart.png")

# ============================================================
# SECTION 15: BEST HYPERPARAMETERS
# ============================================================
print("\n" + "=" * 60)
print("SECTION 14: Best Hyperparameters")
print("=" * 60)

hyp_rows = [{"Model": n, "Best Parameters": str(p)}
            for n, p in best_params_all.items()]
hyp_df = pd.DataFrame(hyp_rows, columns=["Model","Best Parameters"])
print(hyp_df.to_string(index=False))
hyp_df.to_csv(f"{OUTPUT_DIR}/best_hyperparameters.csv", index=False)

# ============================================================
# SECTION 16: TRAINING / VALIDATION LOSS & ACCURACY CURVES
# ============================================================
# Strategy per model type:
#   MLP  → uses loss_curve_ (native per-epoch loss from training)
#          validation loss computed per epoch by re-predicting on X_val
#   Others → sklearn learning_curve() with 8 training-size steps
#             giving train/val accuracy at increasing data fractions
# ============================================================
print("\n" + "=" * 60)
print("SECTION 15: Training / Validation Loss & Accuracy Curves")
print("=" * 60)

# ── 16-A  MLP: native epoch-level curves ──────────────────────────
print("\n  Fitting MLP with epoch-level tracking …")

# Split a 10% validation set from training data for MLP curves
X_tr_mlp, X_val_mlp, y_tr_mlp, y_val_mlp = train_test_split(
    X_train, y_train, test_size=0.1,
    random_state=RANDOM_SEED, stratify=y_train
)

# Build MLP with best hyperparams found in Section 9
mlp_best_p = {k.replace("model__",""):v
              for k,v in best_params_all["MLP"].items()}

# Collect per-epoch loss by training one epoch at a time (warm_start)
mlp_train_loss = []
mlp_val_loss   = []
mlp_train_acc  = []
mlp_val_acc    = []

# Fit a preprocessor on training data first
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.preprocessing import LabelEncoder

prep_only = ColumnTransformer([
    ('num', StandardScaler(),                          numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'),    categorical_features)
])
prep_only.fit(X_tr_mlp)
X_tr_mlp_t  = prep_only.transform(X_tr_mlp)
X_val_mlp_t = prep_only.transform(X_val_mlp)

N_EPOCHS = 100
mlp_epoch = MLPClassifier(
    hidden_layer_sizes = mlp_best_p.get('hidden_layer_sizes', (128,64)),
    activation         = mlp_best_p.get('activation', 'relu'),
    learning_rate_init = mlp_best_p.get('learning_rate_init', 0.001),
    max_iter=1,
    warm_start=True,
    random_state=RANDOM_SEED
)

for epoch in range(N_EPOCHS):
    mlp_epoch.fit(X_tr_mlp_t, y_tr_mlp)

    # Training loss (log-loss)
    tr_prob  = mlp_epoch.predict_proba(X_tr_mlp_t)
    val_prob = mlp_epoch.predict_proba(X_val_mlp_t)
    mlp_train_loss.append(log_loss(y_tr_mlp,  tr_prob))
    mlp_val_loss.append(  log_loss(y_val_mlp, val_prob))

    # Accuracy
    mlp_train_acc.append(accuracy_score(y_tr_mlp,  mlp_epoch.predict(X_tr_mlp_t)))
    mlp_val_acc.append(  accuracy_score(y_val_mlp, mlp_epoch.predict(X_val_mlp_t)))

print(f"  MLP epoch curves collected ({N_EPOCHS} epochs).")

# ── 16-B  All models: sklearn learning_curve (train-size steps) ───
print("\n  Computing learning curves for all 6 models …")

TRAIN_SIZES = np.linspace(0.1, 1.0, 8)
lc_results  = {}          # {name: {train_sizes, train_scores, val_scores}}

for name in pipelines:
    print(f"    {name} …", end=" ", flush=True)
    pipe = best_models[name]          # already fitted; learning_curve re-fits

    train_sizes_abs, train_scores, val_scores = learning_curve(
        estimator   = pipe,
        X           = X_train,
        y           = y_train,
        train_sizes = TRAIN_SIZES,
        cv          = 5,
        scoring     = 'accuracy',
        n_jobs      = -1,
        shuffle     = True,
        random_state= RANDOM_SEED
    )
    lc_results[name] = dict(
        sizes        = train_sizes_abs,
        train_mean   = train_scores.mean(axis=1),
        train_std    = train_scores.std(axis=1),
        val_mean     = val_scores.mean(axis=1),
        val_std      = val_scores.std(axis=1)
    )
    print("done")

# ── 16-C  Plot MLP epoch curves ───────────────────────────────────
epochs = np.arange(1, N_EPOCHS+1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MLP – Training vs Validation Curves (per Epoch)",
             fontsize=14, fontweight='bold')

axes[0].plot(epochs, mlp_train_loss, label='Train Loss',      color='steelblue',  lw=2)
axes[0].plot(epochs, mlp_val_loss,   label='Validation Loss', color='tomato',     lw=2, ls='--')
axes[0].set_xlabel("Epoch");  axes[0].set_ylabel("Log-Loss")
axes[0].set_title("Loss Curve");  axes[0].legend();  axes[0].grid(alpha=0.3)

axes[1].plot(epochs, mlp_train_acc, label='Train Accuracy',      color='steelblue', lw=2)
axes[1].plot(epochs, mlp_val_acc,   label='Validation Accuracy', color='tomato',    lw=2, ls='--')
axes[1].set_xlabel("Epoch");  axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curve");  axes[1].legend();  axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/mlp_epoch_curves.png", dpi=300)
plt.close()
print("\n  Saved: mlp_epoch_curves.png")

# ── 16-D  Plot learning curves for ALL 6 models (2×3 grid) ────────
MODEL_COLORS = {
    "MLP": "#2196F3", "RF": "#4CAF50", "XGB": "#FF9800",
    "LR":  "#9C27B0", "SVM":"#F44336", "KNN": "#00BCD4"
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
fig.suptitle("Learning Curves – Training vs Validation Accuracy\n"
             "(shaded = ±1 std across 5 CV folds)",
             fontsize=14, fontweight='bold')

for i, name in enumerate(pipelines):
    lc = lc_results[name]
    ax = axes[i]
    color = MODEL_COLORS[name]

    ax.plot(lc['sizes'], lc['train_mean'],
            label='Train',      color=color,   lw=2)
    ax.fill_between(lc['sizes'],
                    lc['train_mean'] - lc['train_std'],
                    lc['train_mean'] + lc['train_std'],
                    alpha=0.15, color=color)

    ax.plot(lc['sizes'], lc['val_mean'],
            label='Validation', color=color,   lw=2, ls='--')
    ax.fill_between(lc['sizes'],
                    lc['val_mean'] - lc['val_std'],
                    lc['val_mean'] + lc['val_std'],
                    alpha=0.15, color=color)

    ax.set_title(f"{name}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Training Samples")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/learning_curves_all_models.png", dpi=300)
plt.close()
print("  Saved: learning_curves_all_models.png")

# ── 16-E  Combined learning-curve overlay (all 6 on one plot) ─────
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle("Validation Accuracy Learning Curves – All Models",
             fontsize=13, fontweight='bold')

for name in pipelines:
    lc = lc_results[name]
    ax.plot(lc['sizes'], lc['val_mean'],
            label=name, color=MODEL_COLORS[name], lw=2, marker='o', ms=5)

ax.set_xlabel("Training Samples");  ax.set_ylabel("Validation Accuracy")
ax.set_ylim(0, 1.05);  ax.legend();  ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/learning_curves_overlay.png", dpi=300)
plt.close()
print("  Saved: learning_curves_overlay.png")

# ============================================================
# SECTION 17: BOOTSTRAP WILCOXON APFD COMPARISON
# ============================================================
# Rationale
# ---------
# With a single train-test split we have ONE APFD value per model.
# A Wilcoxon signed-rank test requires paired observations.
# Solution: Bootstrap the held-out test set N_BOOTSTRAP times.
#   → Each bootstrap resample is a "virtual fold"
#   → Compute APFD on each resample for every model
#   → The N_BOOTSTRAP APFD values are paired (same resample per model)
#   → Run Wilcoxon on those paired vectors
#   → Also run Friedman across all 6 models, then Nemenyi post-hoc
# ============================================================
print("\n" + "=" * 60)
print("SECTION 16: Bootstrap Wilcoxon + Friedman APFD Comparison")
print("=" * 60)

rng = np.random.RandomState(RANDOM_SEED)
n_test = len(y_test)

# Cache predicted probabilities (already fitted models)
probs_test = {name: best_models[name].predict_proba(X_test)[:,1]
              for name in pipelines}

# Bootstrap: collect N_BOOTSTRAP APFD scores per model
bootstrap_apfd = {name: [] for name in pipelines}

print(f"\n  Running {N_BOOTSTRAP} bootstrap resamples of test set …", end=" ")
for b in range(N_BOOTSTRAP):
    idx = rng.randint(0, n_test, n_test)          # resample with replacement
    y_b = y_test.values[idx]
    if y_b.sum() == 0:                             # skip all-negative resamples
        continue
    for name in pipelines:
        prob_b = probs_test[name][idx]
        bootstrap_apfd[name].append(compute_apfd(y_b, prob_b))

# Align lengths (in case any resample was skipped)
min_len = min(len(v) for v in bootstrap_apfd.values())
for name in pipelines:
    bootstrap_apfd[name] = bootstrap_apfd[name][:min_len]

print(f"done ({min_len} valid resamples).")

# ── 17-A  Friedman test across all 6 models ───────────────────────
from scipy.stats import friedmanchisquare
boot_matrix = [bootstrap_apfd[n] for n in pipelines]   # 6 × min_len
friedman_stat, friedman_p = friedmanchisquare(*boot_matrix)

print(f"\n  Friedman Test (all 6 models):")
print(f"    χ² = {friedman_stat:.4f},  p = {friedman_p:.6f}")
print(f"    {'Significant difference exists.' if friedman_p < 0.05 else 'No significant difference.'}")

# ── 17-B  Nemenyi post-hoc (pairwise) via scikit_posthocs ─────────
boot_df = pd.DataFrame(
    {name: bootstrap_apfd[name] for name in pipelines}
)

nemenyi_pmat = sp.posthoc_nemenyi_friedman(boot_df.values)
nemenyi_pmat.index   = list(pipelines.keys())
nemenyi_pmat.columns = list(pipelines.keys())

print("\n  Nemenyi Post-Hoc p-value Matrix (pairwise):")
print(nemenyi_pmat.round(4).to_string())
nemenyi_pmat.round(4).to_csv(f"{OUTPUT_DIR}/nemenyi_pmat.csv")

# ── 17-C  Wilcoxon: MLP vs each other model ───────────────────────
mlp_boot = np.array(bootstrap_apfd["MLP"])
comparators = ["RF","XGB","LR","SVM","KNN"]
ALPHA = 0.05

wilcoxon_rows = []
print(f"\n  Wilcoxon Signed-Rank (MLP vs each baseline, α={ALPHA}):")
print(f"  {'Pair':<15} {'MLP µ':>8} {'Other µ':>9} "
      f"{'W':>10} {'p-value':>10} {'Sig?':>6} {'r':>8} {'Direction':>13}")
print("  " + "-" * 85)

for other in comparators:
    other_boot = np.array(bootstrap_apfd[other])
    diffs = mlp_boot - other_boot

    if np.all(diffs == 0):
        stat, p_val = np.nan, 1.0
    else:
        stat, p_val = wilcoxon(mlp_boot, other_boot,
                               alternative='two-sided', zero_method='wilcox')

    sig  = "Yes" if p_val < ALPHA else "No"
    n    = len(mlp_boot)
    r    = float(1 - (2*stat)/(n*(n+1))) if not np.isnan(stat) else np.nan
    mu1  = mlp_boot.mean()
    mu2  = other_boot.mean()
    dirn = "MLP >" if mu1 > mu2 else ("MLP <" if mu1 < mu2 else "Tie")

    wilcoxon_rows.append(dict(
        Comparison=f"MLP vs {other}",
        MLP_Mean=round(mu1,4), Other_Mean=round(mu2,4),
        W_stat=round(float(stat),4) if not np.isnan(stat) else "N/A",
        p_value=round(p_val,6),
        Significant=sig,
        Effect_r=round(r,4) if not np.isnan(r) else "N/A",
        Direction=dirn
    ))

    print(f"  MLP vs {other:<8} {mu1:>8.4f} {mu2:>9.4f} "
          f"{stat:>10.1f} {p_val:>10.6f} {sig:>6} "
          f"{r:>8.4f} {dirn:>13}")

wilcoxon_df = pd.DataFrame(wilcoxon_rows)
wilcoxon_df.to_csv(f"{OUTPUT_DIR}/wilcoxon_bootstrap_results.csv", index=False)
print(f"\n  Note: |r| < 0.3 small | 0.3–0.5 medium | > 0.5 large effect.")
print("  Saved: wilcoxon_bootstrap_results.csv")

# ── 17-D  Bootstrap APFD distribution boxplot ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"Bootstrap APFD Distributions ({min_len} resamples)\n"
             "Used for Wilcoxon Signed-Rank & Friedman Tests",
             fontsize=13, fontweight='bold')

colors_list = [MODEL_COLORS[n] for n in pipelines]
data_list   = [bootstrap_apfd[n] for n in pipelines]
labels_list = list(pipelines.keys())

# Box plot
bp = axes[0].boxplot(data_list, patch_artist=True, notch=True,
                     medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], colors_list):
    patch.set_facecolor(color);  patch.set_alpha(0.7)
axes[0].set_xticklabels(labels_list, fontsize=11)
axes[0].set_ylabel("APFD Score")
axes[0].set_title("Box Plot")
axes[0].set_ylim(0, 1.05);  axes[0].grid(axis='y', alpha=0.3)

# Violin plot
vp = axes[1].violinplot(data_list, showmedians=True, showextrema=True)
for i, (body, color) in enumerate(zip(vp['bodies'], colors_list)):
    body.set_facecolor(color);  body.set_alpha(0.6)
axes[1].set_xticks(np.arange(1, len(labels_list)+1))
axes[1].set_xticklabels(labels_list, fontsize=11)
axes[1].set_ylabel("APFD Score")
axes[1].set_title("Violin Plot")
axes[1].set_ylim(0, 1.05);  axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/bootstrap_apfd_distribution.png", dpi=300)
plt.close()
print("  Saved: bootstrap_apfd_distribution.png")

# ── 17-E  Nemenyi p-value heatmap ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.eye(len(pipelines), dtype=bool)
sns.heatmap(nemenyi_pmat.astype(float),
            annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5,
            mask=mask, ax=ax,
            annot_kws={"size":10})
ax.set_title("Nemenyi Post-Hoc p-value Heatmap\n"
             "(green = significant difference, red = no difference)",
             fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/nemenyi_heatmap.png", dpi=300)
plt.close()
print("  Saved: nemenyi_heatmap.png")

# ── 17-F  Bootstrap 95% CI summary table ──────────────────────────
ci_rows = []
for name in pipelines:
    arr = np.array(bootstrap_apfd[name])
    ci_rows.append(dict(
        Model  = name,
        APFD   = round(results[name]['apfd'], 4),
        Boot_Mean = round(arr.mean(), 4),
        CI_Lower  = round(np.percentile(arr, 2.5), 4),
        CI_Upper  = round(np.percentile(arr, 97.5), 4)
    ))

ci_df = pd.DataFrame(ci_rows)
print("\n  Bootstrap 95% CI for APFD:")
print(ci_df.to_string(index=False))
ci_df.to_csv(f"{OUTPUT_DIR}/bootstrap_apfd_ci.csv", index=False)
print("  Saved: bootstrap_apfd_ci.csv")

# ── 17-G  CI bar chart ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,6))
x_pos = np.arange(len(ci_df))
ax.bar(x_pos, ci_df['Boot_Mean'], color=colors_list,
       edgecolor='black', alpha=0.8, label='Bootstrap Mean APFD')

# Error bars = 95% CI
lower_err = ci_df['Boot_Mean'] - ci_df['CI_Lower']
upper_err = ci_df['CI_Upper']  - ci_df['Boot_Mean']
ax.errorbar(x_pos, ci_df['Boot_Mean'],
            yerr=[lower_err, upper_err],
            fmt='none', color='black', capsize=6, lw=2)

ax.set_xticks(x_pos)
ax.set_xticklabels(ci_df['Model'], fontsize=12)
ax.set_ylabel("APFD Score");  ax.set_ylim(0, 1.1)
ax.set_title("Bootstrap Mean APFD with 95% Confidence Intervals",
             fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/apfd_ci_bar_chart.png", dpi=300)
plt.close()
print("  Saved: apfd_ci_bar_chart.png")

# ============================================================
# SECTION 18: SAVE ALL RESULTS TO JSON & EXCEL
# ============================================================
with open(f"{OUTPUT_DIR}/final_results.json","w") as f:
    json.dump(results, f, indent=4)

# Compile all tables into one Excel workbook
excel_path = f"{OUTPUT_DIR}/all_results.xlsx"
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Add this line alongside the other .to_excel() calls
    pd.DataFrame([{
    'Model': name,
    'Accuracy': round(default_results[name]['accuracy'], 4),
    'Precision': round(default_results[name]['precision'], 4),
    'Recall': round(default_results[name]['recall'], 4),
    'F1_Score': round(default_results[name]['f1'], 4),
    'APFD': round(default_results[name]['apfd'], 4)
} for name in default_pipelines]).to_excel(writer, sheet_name="Default_Performance", index=False)
    # Performance table (clean numeric version)
    perf_rows = []
    for name in pipelines:
        r = results[name]
        perf_rows.append(dict(Model=name,
            Accuracy=round(r['accuracy'],4),  Precision=round(r['precision'],4),
            Recall=round(r['recall'],4),       F1_Score=round(r['f1'],4),
            APFD=round(r['apfd'],4)))
    pd.DataFrame(perf_rows).to_excel(writer, sheet_name="Performance",   index=False)
    wilcoxon_df.to_excel(            writer, sheet_name="Wilcoxon",       index=False)
    ci_df.to_excel(                  writer, sheet_name="Bootstrap_CI",   index=False)
    nemenyi_pmat.round(4).to_excel(  writer, sheet_name="Nemenyi_Posthoc")
    hyp_df.to_excel(                 writer, sheet_name="Hyperparameters", index=False)

print(f"\n  Saved: all_results.xlsx  (5 sheets)")

print(f"\n{'='*60}")
print(f"All outputs saved to '{OUTPUT_DIR}/'")
print(f"{'='*60}")
print("Done!")

SECTION 1: Loading Data
Raw shape: (64024, 8)
SECTION 2: Preprocessing
After cleaning: (38808, 5)
Duration log-transformed.

Class distribution:
0.0    28215
1.0    10593
Name: Verdict, dtype: int64
Fail rate: 27.3%

SECTION 3: Feature Setup
Numeric features:     ['Cycle', 'Duration_log']
Categorical features: ['LastRun', 'LastResults']
SECTION 4: Train-Test Split (80/20 Stratified)
Training set : (31046, 4)
Test set     : (7762, 4)
Train fail rate : 27.3%
Test  fail rate : 27.3%

SECTION 7: Defining Models
SECTION 9: Training with GridSearchCV (cv=5 on train set)

Training MLP...
  Best params: {'model__activation': 'relu', 'model__hidden_layer_sizes': (64, 32), 'model__learning_rate_init': 0.001}
  Accuracy=0.9977 | Precision=1.0000 | Recall=0.9915 | F1=0.9957 | APFD=0.8635

Training RF...
  Best params: {'model__max_depth': None, 'model__n_estimators': 200}
  Accuracy=0.9956 | Precision=0.9901 | Recall=0.9939 | F1=0.9920 | APFD=0.8634

Training XGB...
  Best params: {'model__learnin